<a href="https://colab.research.google.com/github/Hem1144/AI-ML/blob/ml-labs/Word2VecImplementationAndComparisonWeek7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Modified Lab 2 with Word2Vec Implementation and Comparison

In [1]:
from sklearn.datasets import fetch_20newsgroups
from pyspark.sql import SparkSession
from pyspark.ml.feature import (Tokenizer, HashingTF, IDF, Str  ingIndexer,
                               StopWordsRemover, Word2Vec)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pandas as pd

# Start Spark Session
spark = SparkSession.builder.appName("DocumentClassification").getOrCreate()

# Fetch 20 Newsgroups Data
newsgroups = fetch_20newsgroups(subset='all')

# Convert the dataset to a DataFrame for PySpark processing
data = pd.DataFrame({'text': newsgroups.data, 'category': newsgroups.target})
df = spark.createDataFrame(data)

# Filter 25% of documents from each category
df_sampled = df.sample(withReplacement=False, fraction=0.25, seed=42)

# Split the data into training and testing sets (80% train, 20% test)
train_data, test_data = df_sampled.randomSplit([0.8, 0.2], seed=42)

# Preprocessing steps common to both models
tokenizer = Tokenizer(inputCol="text", outputCol="words")
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
indexer = StringIndexer(inputCol="category", outputCol="label")

# ===== TF-IDF Pipeline =====
hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=1000)
idf = IDF(inputCol="raw_features", outputCol="features")
lr_tfidf = LogisticRegression(featuresCol="features", labelCol="label")

tfidf_pipeline = Pipeline(stages=[
    tokenizer,
    stopwords_remover,
    hashingTF,
    idf,
    indexer,
    lr_tfidf
])

# ===== Word2Vec Pipeline =====
word2Vec = Word2Vec(vectorSize=100, minCount=1, inputCol="filtered_words", outputCol="features_w2v")
lr_w2v = LogisticRegression(featuresCol="features_w2v", labelCol="label")

w2v_pipeline = Pipeline(stages=[
    tokenizer,
    stopwords_remover,
    word2Vec,
    indexer,
    lr_w2v
])

# Train and evaluate TF-IDF model
print("Training TF-IDF model...")
tfidf_model = tfidf_pipeline.fit(train_data)
tfidf_predictions = tfidf_model.transform(test_data)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
tfidf_accuracy = evaluator.evaluate(tfidf_predictions)
print(f"TF-IDF Model Accuracy: {tfidf_accuracy:.2f}")

# Train and evaluate Word2Vec model
print("\nTraining Word2Vec model...")
w2v_model = w2v_pipeline.fit(train_data)
w2v_predictions = w2v_model.transform(test_data)

w2v_accuracy = evaluator.evaluate(w2v_predictions)
print(f"Word2Vec Model Accuracy: {w2v_accuracy:.2f}")

# Compare the results
print("\nComparison:")
print(f"TF-IDF Accuracy: {tfidf_accuracy:.2f}")
print(f"Word2Vec Accuracy: {w2v_accuracy:.2f}")

if tfidf_accuracy > w2v_accuracy:
    print("\nTF-IDF performed better than Word2Vec for this dataset.")
elif w2v_accuracy > tfidf_accuracy:
    print("\nWord2Vec performed better than TF-IDF for this dataset.")
else:
    print("\nBoth models performed equally well.")

# Show some predictions from both models
print("\nSample TF-IDF predictions:")
tfidf_predictions.select("text", "category", "prediction").show(5, truncate=False)

print("\nSample Word2Vec predictions:")
w2v_predictions.select("text", "category", "prediction").show(5, truncate=False)

# Stop Spark Session
spark.stop()

Training TF-IDF model...
TF-IDF Model Accuracy: 0.53

Training Word2Vec model...
Word2Vec Model Accuracy: 0.44

Comparison:
TF-IDF Accuracy: 0.53
Word2Vec Accuracy: 0.44

TF-IDF performed better than Word2Vec for this dataset.

Sample TF-IDF predictions:
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------